## Descomprimir archivos con imagenes

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1Ws5mN7BejJwrCgUtoMUhpk-VVLqWD5rX?usp=sharing
!unzip /content/Dataset//content/Dataset/garbage.zip

## Librerías

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, classification_report

## Configuración básica

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# Fijar semilla para reproducibilidad
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

## Rutas conjunto de datos(train-test)

In [ ]:
train_dir = "/content/dataset_perro_gato/dataset/training_set"
test_dir = "/content/dataset_perro_gato/dataset/test_set"

## Análisis de conjunto de datos(train)

In [ ]:
classes = sorted(os.listdir(train_dir))
num_images = []
for cls in classes:
    cls_path = os.path.join(train_dir, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        num_images.append(count)

plt.figure(figsize=(12,5))
plt.bar(range(len(classes)), num_images)
plt.xticks(range(len(classes)), classes, rotation=90)
plt.xlabel("Clases")
plt.ylabel("Cantidad de imágenes")
plt.title("Distribución de imágenes por clase - Entrenamiento")
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, count in enumerate(num_images):
    plt.text(i, count + 20, str(count), ha='center', va='bottom', fontsize=8, rotation=90)

plt.ylim(0, max(num_images) * 1.15)

plt.tight_layout()
plt.show()

## Normalizar datos de entrenamiento y prueba

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

## Dividir conjunto de datos

In [ ]:
# Crear los Datasets usando las carpetas y transformaciones correctas
train_dataset = ImageFolder(root=train_dir, transform=data_transforms)
test_dataset = ImageFolder(root=test_dir, transform=data_transforms)

# Crear los DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Número de imágenes de entrenamiento: {len(train_dataset)}")
print(f"Número de imágenes de prueba: {len(test_dataset)}")
print(f"Número de lotes en train_loader: {len(train_loader)}")
print(f"Número de lotes en test_loader: {len(test_loader)}")
print(f"Clases encontradas: {train_dataset.classes}")

## Cargar modelo preentrenado ResNet50

In [ ]:
model = models.resnet50(weights="IMAGENET1K_V1")

In [ ]:
model.eval()

## Adaptar número de clases

In [ ]:
num_classes = len(classes)
print("Numero de clases:", num_classes)

# Adaptar la última capa fully connected al número de clases del dataset
model.fc = nn.Linear(model.fc.in_features, num_classes)

## Congelar capas

In [ ]:
# Congelar todas las capas excepto layer4 y fc
for name, param in model.named_parameters():
    if "fc" in name or "layer4" in name:
        param.requires_grad = True  # Capa entrenable
    else:
        param.requires_grad = False # Capas congeladas

# Verificar qué capas se están entrenando
print("Parámetros que se están entrenando:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

## Definir función de pérdida y optimizador

In [ ]:
criterion = nn.CrossEntropyLoss()

trainable_params = filter(lambda p: p.requires_grad, model.parameters())

optimizer = optim.Adam(trainable_params, lr=1e-3)

## Entrenamiento

In [ ]:
epochs = 10
model = model.to(device)

best_test_acc = 0.0
save_path = "resnet50_best.pt"
start = time.time()

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()                # Reiniciar gradientes
        outputs = model(imgs)                # Forward pass
        loss = criterion(outputs, labels)    # Calcular pérdida
        loss.backward()                      # Backpropagation
        optimizer.step()                     # Actualizar pesos

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = total_loss / total

    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc*100:.2f}% | "
          f"Test Acc: {test_acc*100:.2f}%")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), save_path)
        print(f"Mejor modelo guardado en '{save_path}' (Test Acc: {best_test_acc*100:.2f}%)")

end = time.time()
print(f"Tiempo de entrenamiento: {(end-start)/60:.3f} minutos")
print(f"Mejor accuracy de prueba: {best_test_acc*100:.2f}%")

## Matriz de confusión

In [ ]:
all_preds, all_labels = [], []

# No calcular gradientes (modo evaluación)
with torch.no_grad():
    for imgs, labels in test_loader: # Iterar sobre batches de validación
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, cmap="Blues", annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
plt.title("Matriz de Confusión")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

## Reporte de clasificación

In [ ]:
# Reporte por clase
print(classification_report(all_labels, all_preds, target_names=classes, digits=3))

## Clasificar una imagen

In [ ]:
img_path = ""

pil_img = Image.open(img_path)

inference_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

img_tensor = inference_transforms(pil_img).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    outputs = model(img_tensor)
    probs = torch.softmax(outputs, dim=1)
    pred_class_idx = torch.argmax(probs, dim=1).item()
    confidence = probs[0, pred_class_idx].item()

predicted_class_name = classes[pred_class_idx]
print(f"Predicción: {predicted_class_name} con confianza {confidence*100:.2f}%")

plt.imshow(pil_img)
plt.title(f"Predicción: {predicted_class_name} ({confidence*100:.1f}%)")
plt.axis("off")
plt.show()